# 02 — Train ViT & ResNet and compare data efficiency

**Requires GPU** on Colab for reasonable ViT training time.

Install deps (Colab):
```bash
pip install -q -r ../requirements.txt
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

from src.data import (
    load_all_samples,
    make_loaders,
    resolve_data_root,
    split_train_val,
    subset_train_stratified,
)
from src.evaluate import evaluate_model
from src.metrics import format_metrics_row
from src.train_resnet import train_resnet
from src.train_vit import train_vit
from src.utils import ensure_dir, load_config, resolve_device, set_seed
from src.visualize import plot_confusion_matrix, plot_training_history, save_gradcam_examples

cfg = load_config(ROOT / 'configs' / 'default.yaml')
set_seed(cfg['seed'])
device = resolve_device(cfg.get('device', 'auto'))
print('Device:', device)

In [ ]:
data_root = resolve_data_root(cfg)
train_pool, test_samples, layout = load_all_samples(data_root, layout=cfg['data'].get('layout', 'auto'))
train_samples, val_samples = split_train_val(
    train_pool,
    val_fraction=cfg['data']['val_fraction'],
    seed=cfg['seed'],
    group_augmentations=cfg['data'].get('group_augmentations', True),
)

FRACTION = 0.25  # change to 0.5 or 1.0 for other runs
MODEL = 'resnet'  # or 'vit'

train_subset = subset_train_stratified(
    train_samples,
    fraction=FRACTION,
    seed=cfg['seed'],
    group_augmentations=cfg['data'].get('group_augmentations', True),
)
train_loader, val_loader, test_loader = make_loaders(
    train_subset,
    val_samples,
    test_samples,
    image_size=cfg['data']['image_size'],
    batch_size=cfg['data']['batch_size'],
)
print(f'Model={MODEL} fraction={FRACTION} n_train={len(train_subset)}')

In [ ]:
run_name = f"{MODEL}_frac{FRACTION}_seed{cfg['seed']}"
ckpt_path = ensure_dir(ROOT / cfg['paths']['checkpoints'] / run_name) / 'best.pt'

if MODEL == 'resnet':
    model, train_meta, history = train_resnet(train_loader, val_loader, device, cfg, ckpt_path)
    is_vit = False
else:
    model, train_meta, history = train_vit(train_loader, val_loader, device, cfg, ckpt_path)
    is_vit = True

eval_out = evaluate_model(model, test_loader, device, is_vit=is_vit)
row = format_metrics_row(
    model=MODEL,
    train_fraction=FRACTION,
    metrics=eval_out['metrics'],
    train_time_s=train_meta['train_time_s'],
    infer_latency_ms=eval_out['infer_latency_ms'],
    model_size_mb=eval_out['model_size_mb'],
    seed=cfg['seed'],
)
display(pd.DataFrame([row]))

In [ ]:
fig_dir = ensure_dir(ROOT / cfg['paths']['figures'] / run_name)
plot_training_history(history, fig_dir / 'training_curves.png', title=run_name)
plot_confusion_matrix(eval_out['metrics']['confusion_matrix'], fig_dir / 'confusion_matrix.png', title=run_name)
print('Saved figures to', fig_dir)

if MODEL == 'resnet':
    save_gradcam_examples(model, test_loader, fig_dir / 'gradcam.png', device)

## Full grid via CLI (recommended)

From repo root:
```bash
for m in resnet vit; do for f in 0.25 0.5 1.0; do python scripts/run_experiment.py --model $m --fraction $f; done; done
```
Then load `results/comparison_table.csv` for thesis tables.